In [1]:
!pip install yfinance

^C


VaR Modeling - Historical simulation VaR

In [212]:
import yfinance as yf 
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

In [83]:
def extract_data(underlyings_list,valuation_date,look_back_period):
    # Default looking back period: 2 years 
    end_date  =  (datetime.strptime(valuation_date, "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d")
    start_date = (datetime.strptime(end_date, "%Y-%m-%d") - timedelta(days=365*look_back_period)).strftime("%Y-%m-%d")
    extract_list = []
    for values in underlyings_list:
        data = yf.download(values, start=start_date, end=end_date, interval="1d")
        # data = yf.download(values, period = '2y', interval ='1d')
        data = data['Close'][[values]].reset_index()
        data.columns = ['Date',str(values) + "_close"]
        extract_list.append(data)
        # print(data) 
    combined_df = extract_list[0].copy()
    for df in extract_list[1:]:
        combined_df = pd.merge(combined_df, df, on='Date', how='inner')
    combined_df = combined_df.sort_values('Date').reset_index(drop=True)
    return combined_df

In [85]:
def calculate_HVaR(underlyings_list,portfolio_share_list,time_horizon,alpha,extract_df):
    extract_df = extract_df.copy()
    valuation_date_df = extract_df.tail(1).copy()
    valuation_date_df.rename(columns={col: f'mtm_{col}' for col in valuation_date_df.columns}, inplace=True)
    # mtm_price_list = valuation_date_df.columns[1:].values
    mtm_price_list = [col for col in valuation_date_df.columns if '_close' in col]
    # print(mtm_price_list)
    # print(valuation_date_df)
    # print(portfolio_share_list)
    valuation_date_df['mtm_portfolio_value'] = valuation_date_df[mtm_price_list].dot(portfolio_share_list)

    for values in underlyings_list:
        columns_name = values+'_'+str(time_horizon)+'D_changes'
        extract_df[columns_name] = extract_df[str(values) + "_close"].shift(-time_horizon)/ extract_df[str(values) + "_close"]
        # print(columns_name)

    extract_df_new = extract_df.assign(**valuation_date_df.squeeze())

    simulated_price_list = []
    for values in underlyings_list:
        columns_name = values+'_'+ "simulated_price"
        extract_df_new[columns_name] = extract_df_new[values+'_'+str(time_horizon)+'D_changes']*extract_df_new["mtm_"+values+'_close'] 
        simulated_price_list.append(columns_name)
    extract_df_new['simulated_portfolio_value'] = extract_df_new[simulated_price_list].dot(portfolio_share_list)
    extract_df_new['simulated_pnl'] = extract_df_new['simulated_portfolio_value'] - extract_df_new['mtm_portfolio_value']
    
    extract_df_new_final = extract_df_new[['simulated_pnl']]
    extract_df_new_final= extract_df_new_final.dropna(subset=['simulated_pnl'])
    extract_df_new_final = extract_df_new_final.sort_values(by='simulated_pnl', ascending=True)
    HVaR = extract_df_new_final['simulated_pnl'].quantile(alpha)
    return HVaR

In [87]:
######################
# Procedure 
# Step 1: Input all the ticket names of your portfolio into underlyings_list. Remark: Ensure the ticket name is the same as Yahoo Finance setup
# step 2: input all the shares' amounts of each underlying under your portfolio in portfolio_share_list
# step 3: input the valuation date(default is today's date) and the look back period on historical data (default is 2 years)
# step 4: input the time horizon and alpha to customize your HVaR calculation 

# calculate_HVaR(underlyings_list, portfolio_share_list. time_horizon, alpha ,extract_df)
# time_horizon = eg 1 day, 10 days, 30 days
# alpha = 1 - confidence interval eg 99% -> alpha:0.01, 95% -> alpha: 0.05

######################

In [150]:
underlyings_list = ["COIN","NFLX","RACE"]
portfolio_share_list = [24,140,12]
# valuation_date = "2025-12-31" 
valuation_date = str(datetime.today().strftime("%Y-%m-%d"))
look_back_period = 2
extract_df = extract_data(underlyings_list,valuation_date,look_back_period)
HVaR_30D_95 = calculate_HVaR(underlyings_list, portfolio_share_list, 30, 0.05 , extract_df)
HVaR_10D_95 = calculate_HVaR(underlyings_list, portfolio_share_list, 10, 0.05 , extract_df)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [160]:
print(f"Valuation Date: {str(datetime.today())}")
print(f"Primary risk indictor   - 95% HVaR(30D) is :{HVaR_30D_95}")
print(f"Secondary risk indictor - 95% HVaR(10D) is :{HVaR_10D_95}")

Valuation Date: 2026-01-03 01:22:17.491984
Primary risk indictor   - 95% HVaR(30D) is :-3315.237664072695
Secondary risk indictor - 95% HVaR(10D) is :-2155.3421463484838
